# 🛢️ SpillSense AI — Full 150-Epoch Training on Google Colab

### ✅ Fully automated — just click Run All!
- Code clones from **GitHub** automatically
- Dataset downloads from **Zenodo** automatically (~1.1 GB, ~1-2 min)
- Results save to **Google Drive** automatically
- **Zero manual uploads needed!**

### Before you start:
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Click **Runtime → Run all**
3. Allow Google Drive access when prompted
4. Sit back and wait ~1-2 hours ☕

---
## 📋 Step 1: Check GPU

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    mem_gb = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f'GPU Memory: {mem_gb:.1f} GB')
    print('\n✅ GPU is ready!')
else:
    print('\n❌ NO GPU! Go to Runtime → Change runtime type → T4 GPU')

---
## 📦 Step 2: Clone Code from GitHub

In [ ]:
import os

GITHUB_REPO = 'https://github.com/vedhikaa25/oil-spill-detector-.git'
PROJECT_DIR = '/content/SpillSense-AI'

if os.path.exists(PROJECT_DIR):
    print('Project already exists, pulling latest...')
    !cd "$PROJECT_DIR" && git pull
else:
    print(f'Cloning from {GITHUB_REPO}...')
    !git clone "$GITHUB_REPO" "$PROJECT_DIR"

os.chdir(PROJECT_DIR)
print(f'\n✅ Code ready at: {PROJECT_DIR}')
print(f'Files: {[f for f in os.listdir(".") if not f.startswith(".")]}')

---
## 📥 Step 3: Download Dataset from Zenodo (~1-2 min)

The 1.1 GB dataset downloads directly from Zenodo — Colab's internet is fast!

In [ ]:
import os
import time

DATASET_URL = 'https://zenodo.org/api/records/10555314/files/dataset.zip/content'
DATASET_ZIP = '/content/dataset.zip'
DATASET_DIR = f'{PROJECT_DIR}/dataset'

if os.path.exists(DATASET_DIR) and len(os.listdir(DATASET_DIR)) > 1:
    print('✅ Dataset already exists, skipping download.')
else:
    print('📥 Downloading Oil Spill Drone Dataset from Zenodo (~1.1 GB)...')
    print('   Colab has fast internet — this takes ~1-2 minutes\n')
    
    start = time.time()
    !wget -q --show-progress -O "$DATASET_ZIP" "$DATASET_URL"
    dl_time = time.time() - start
    print(f'\n   Downloaded in {dl_time:.0f} seconds')
    
    print('\n📦 Extracting dataset...')
    os.makedirs(DATASET_DIR, exist_ok=True)
    !unzip -q -o "$DATASET_ZIP" -d "$PROJECT_DIR/"
    os.remove(DATASET_ZIP)
    print('🗑️ Cleaned up zip file')

# Verify
print(f'\n📊 Dataset:')
for split in ['train', 'val', 'test']:
    img_dir = f'{DATASET_DIR}/{split}/images'
    mask_dir = f'{DATASET_DIR}/{split}/masks'
    if os.path.exists(img_dir):
        n_img = len(os.listdir(img_dir))
        n_mask = len(os.listdir(mask_dir)) if os.path.exists(mask_dir) else 0
        print(f'  {split}: {n_img} images, {n_mask} masks ✅')
    else:
        print(f'  {split}: NOT FOUND ❌')

---
## 🔧 Step 4: Install Dependencies

In [ ]:
!pip install -q segmentation-models-pytorch albumentations

import config
print(f'\nDevice: {config.DEVICE}')
print(f'Max epochs: {config.MAX_EPOCHS}')
print(f'Batch size: {config.BATCH_SIZE}')
print(f'Image size: {config.IMAGE_SIZE}')
print(f'\n✅ Dependencies installed!')

---
## 💾 Step 5: Mount Google Drive (for saving results)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_SAVE_DIR = '/content/drive/MyDrive/SpillSense-AI-Results'
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
print(f'\n✅ Results will auto-save to: Google Drive → SpillSense-AI-Results/')

---
## 🚀 Step 6: Train! (150 Epochs, Early Stopping)

This starts **fresh full training** (150 epochs max, early stopping patience=10).

**Estimated time: ~1-2 hours on T4 GPU**

In [ ]:
# Create necessary directories
os.makedirs('models', exist_ok=True)
os.makedirs('outputs', exist_ok=True)
os.makedirs('logs', exist_ok=True)

print('🚀 Starting full training...\n')
!python train.py

---
## 💾 Step 7: Save Results to Google Drive

In [ ]:
import shutil

print('💾 Saving results to Google Drive...\n')

save_items = {
    'models/best_model.pth': 'best_model.pth',
    'logs/training_log.csv': 'training_log.csv',
    'outputs/training_curves.png': 'training_curves.png',
}

for src, dst in save_items.items():
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(DRIVE_SAVE_DIR, dst))
        size_mb = os.path.getsize(src) / (1024**2)
        print(f'  ✅ {src} → Drive ({size_mb:.1f} MB)')
    else:
        print(f'  ⚠️ {src} not found')

print(f'\n📁 Saved to: Google Drive → SpillSense-AI-Results/')

---
## 📊 Step 8: Evaluate on Test Set

In [ ]:
!python evaluate.py

---
## 📈 Step 9: View Training Curves

In [ ]:
from IPython.display import Image, display
if os.path.exists('outputs/training_curves.png'):
    display(Image(filename='outputs/training_curves.png', width=900))
else:
    print('Training curves not found.')

---
## 🖼️ Step 10: View Test Predictions

In [ ]:
import glob
from IPython.display import Image, display

preds = sorted(glob.glob('outputs/test_predictions/*.png'))[:5]
if preds:
    for p in preds:
        print(os.path.basename(p))
        display(Image(filename=p, width=900))
else:
    print('No test predictions found — run evaluate.py first.')

---
## 📥 Step 11: Download Results

In [ ]:
import shutil
from google.colab import files

# Save test predictions to Drive
test_pred_src = 'outputs/test_predictions'
test_pred_dst = os.path.join(DRIVE_SAVE_DIR, 'test_predictions')
if os.path.exists(test_pred_src) and os.listdir(test_pred_src):
    if os.path.exists(test_pred_dst):
        shutil.rmtree(test_pred_dst)
    shutil.copytree(test_pred_src, test_pred_dst)

# Zip and download
!zip -r /content/spillsense_results.zip models/ outputs/ logs/

print('\n📥 Downloading results...')
files.download('/content/spillsense_results.zip')

---
## ✅ Done!

### On your laptop, replace these files:
```
SpillSense-AI/
├── models/best_model.pth       ← cloud-trained model
├── logs/training_log.csv       ← full training log
└── outputs/
    ├── training_curves.png     ← full curves
    └── test_predictions/       ← predictions
```

Then run: `python predict.py --image path/to/image.jpg`

### If Colab disconnected:
Your results are safe on **Google Drive → SpillSense-AI-Results/**